Q1.
(a) None
(b) '2026-05-06'
(c) ['2026-05-06', '2026-05-18']
(d) [('2026', '05', '06') ('2026', '05', '28')]
(e) ['2026-05-06', '2026-05-18']
추가 질문: c의 경우 캡처가 없으므로 찾고자 한 형태 그대로 돌려주고, d의 경우 캡처 그룹을 사용했으므로 각각의 값이 따로 기억되어 튜플 형식의 반환값이 돌아온다. e의 경우 비캡처 그룹을 사용했기 때문에 각각의 값이 따로 기억되지 않고 찾고자 한 형태 그대로 돌려준다.

Q2.
(a) '[T]!'
(b) '[T]안녕[T] [T]세상[T]!'
(c) '[T]안녕[T] [T]세상[T]!'
(d) '수강생 <30>명, 조교 <3>명'
(e) '수강생 <\x01>명, 조교 <\x01>명'

추가질문
1: (a)는 탐욕적 매칭이므로 .+를 가능한 한 길게 매칭하려고 하고, (b)는 게으른 매칭이므로 .+를 가능한 한 짧게 매칭하려고 하기 때문이다.
2: (d)의 경우 원시 문자열을 사용해 "\1"이 역참조로 제대로 기능했지만, (e)의 경우 일반 문자열로 작성해 "\1"이 8진수 이스케이프로 해석되었기 때문이다.

In [21]:
#Q3 - 강의 SNS 게시물 분석기

posts: list[str] = [
    "오늘 #파이썬 수업 진짜 재밌었음!! @prof_kim @hong 감사 ㅎㅎ"
    "자료: https://etl.snu.ac.kr/lec17",
    "@lee @park 팀플 어디서 모이지ㅠㅠ #DCCP2026 #팀플 카톡 ㄱㄱ",
    "<b>중요</b>: 다음 시험 범위는 1-15강. "
    "문의는 mam3b@snu.ac.kr (010-1234-5678)로!",
    "   여러    공백과\n\n\n줄바꿈이    많은    텍스트  ",
    "ㅋㅋㅋ #파이썬 진짜 좋다 #추천 https://snu.ac.kr",
]

import re
from collections import Counter

email = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
phone = re.compile(r"(\d{2,4}-\d{3,4}-\d{4})")
hashtag = re.compile(r"#(\w+)")

def clean_post(post: str) -> str:
    post = re.sub(r"https?://\S+", " ", post)
    post = re.sub(r"<.+?>", "", post)
    post = email.sub("[이메일]", post)
    post = phone.sub("[전화]", post)
    post = re.sub(r"@\w+", " ", post)
    post = hashtag.sub(" ", post)
    post = re.sub(r"[\u3131-\u3163]+", " ", post)
    post = re.sub(r"\s+", " ", post).strip()
    return post

def extract_hashtags(post: str) -> list[str]:
    hashtags = hashtag.findall(post)
    return hashtags

def analyze_posts(posts: list[str]) -> dict:
    posts_n = len(posts)
    h_counts = []
    masked_count = 0
    length = 0
    for i in posts:
        length += len(clean_post(i))
        m = hashtag.findall(i)
        h_counts.extend(m)
        i, e = email.subn("[이메일]", i)
        i, p = phone.subn("[전화]", i)
        masked_count += e+p
    counter = Counter(h_counts)
    hashtag_counters = dict(counter.most_common())
    result = {"posts_n": posts_n, 
              "avg_length_after_clean": round((length)/(posts_n), 2), 
              "hashtag_counts": hashtag_counters,
              "masked_count": masked_count
              }
    return result

for n in posts:
    print(clean_post(n))

analyze_posts(posts)


오늘 수업 진짜 재밌었음!! 감사 자료:
팀플 어디서 모이지 카톡
중요: 다음 시험 범위는 1-15강. 문의는 [이메일] ([전화])로!
여러 공백과 줄바꿈이 많은 텍스트
진짜 좋다


{'posts_n': 5,
 'avg_length_after_clean': 19.4,
 'hashtag_counts': {'파이썬': 2, 'DCCP2026': 1, '팀플': 1, '추천': 1},
 'masked_count': 2}

Q3. 설명
이메일, 전화번호, 해시태그 형식은 해시태그 추출 함수와 게시물 분석 함수에서도 쓰이므로 re.compile로 미리 컴파일해두었다. clean_post에서는 단계 3과 단계 4의 순서를 바꾸면 멘션을 찾을 때 이메일의 @\w+부분까지 잡혀 잘못 마스킹될 수 있으므로 6단계를 반드시 순서대로 작성해야 한다. 해시태그는 컴파일 시에 #을 빼고 그룹화해 해시태그 추출에서 그대로 사용했고, counter을 사용해 빈도를 기록하고 딕셔너리로 만들었다.